In [0]:
%run ../config/config

In [0]:
import json
import pandas as pd
import numpy as np
import os
import sys
import time

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from datetime import datetime

sys.path.insert(0, str(PROJECT_PATH))

from utils.data_drift import DataDriftAnalyzer, DataDrift
from utils.logger import MLOpsLogger
from utils.metadata_serializer import MetadataSerializer

print("Imports completados exitosamente")

In [0]:
logger = MLOpsLogger(
    name='06_quality_outputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': 'engine_drift'}
)

In [0]:
# =============================================================================
# CONFIGURACIÓN CENTRALIZADA
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("CARGANDO CONFIGURACIÓN PARA DRIFT")
    logger.info("=" * 60)

    from utils.config_loader import ConfigLoader
    config_loader = ConfigLoader(QUALITY_CONFIG)

    column_settings = config_loader.get_column_settings()
    report_settings = config_loader.get_report_settings()
    drift_thresholds = config_loader.get_drift_thresholds()
    drift_metrics_to_consider = config_loader.get_drift_default_metrics()
    drift_bins = config_loader.get_drift_bins()

    campo_primario = column_settings.get('campo_primario', 'codclaveunicocli')
    columnas_control = column_settings.get('columnas_control', ['codclaveunicocli', 'codclavepartycli', 'codmes', 'fecproceso', 'fecdata'])
    titulo_drift_report = report_settings.get('titulo_drift', 'Reporte de Data Drift - Outputs')

    logger.info("✓ CONFIGURACIÓN DRIFT CARGADA")

except Exception as e:
    logger.log_exception(operation='load_drift_config', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ★ CARGA DESDE DELTA STAGING (plan depth = 1)
# =============================================================================
try:
    logger.log_stage_start('engine_drift', MES_VTA=MES_VTA, environment=ENV)
    start_time = time.time()

    spark = SparkSession.builder.getOrCreate()

    # ★ Leer desde Delta staging - plan limpio, sin recursion
    persist_path = f"{ADLS_LOCATION_TABLE}/quality_engine/{ENV}/{TABLE_OUTPUT}"

    logger.info(f"Leyendo desde Delta staging: {persist_path}")
    load_start = time.time()

    df_staging = spark.read.format("delta").load(persist_path)

    # ★ Filtrar actual e histórico - plan depth = 1
    df_actual = df_staging.filter(F.col("codmes") == MES_VTA).cache()
    actual_count = df_actual.count()

    if actual_count == 0:
        raise ValueError(f"No hay registros para el mes {MES_VTA}")

    df_historico = df_staging.filter(F.col("codmes") < MES_VTA).cache()
    historico_count = df_historico.count()

    meses_historicos = sorted([r.codmes for r in df_historico.select('codmes').distinct().collect()])

    load_duration = time.time() - load_start
    logger.info(f"✓ Actual: {actual_count:,} | Histórico: {historico_count:,} ({len(meses_historicos)} meses) | {load_duration:.1f}s")

except Exception as e:
    logger.log_exception(operation='load_drift_data', exception=e, should_raise=True)

In [0]:
# =============================================================================
# CONFIGURACIÓN DE VARIABLES
# =============================================================================
try:
    todas_columnas = df_actual.columns
    columnas_numericas = [f.name for f in df_actual.schema.fields if isinstance(f.dataType, NumericType)]
    variables_existentes = [c for c in todas_columnas if c not in columnas_control and c in columnas_numericas]

    logger.info(f"Variables a analizar drift: {len(variables_existentes)} numéricas")

    if len(variables_existentes) == 0:
        raise ValueError("No hay variables numéricas para analizar drift")

except Exception as e:
    logger.log_exception(operation='configure_drift_variables', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ANÁLISIS DE DRIFT
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("ANÁLISIS DE DRIFT")
    drift_start = time.time()

    drift_analyzer = DataDriftAnalyzer(df_historico, df_actual, TEMPLATE_REPORT_PATH)

    drift_results = drift_analyzer.run(
        columns=variables_existentes,
        drift_thresholds=drift_thresholds
    )

    drift_duration = time.time() - drift_start

    drift_results_with_summary = None
    semaphore_kpis = None
    semaphore_config_drift = None
    variable_comparison = None
    significant_drift = 0
    avg_psi = 0
    max_psi = 0

    if drift_results is not None and not drift_results.empty:
        logger.info(f"✓ Drift en {drift_duration:.2f}s, {len(drift_results)} variables")

        drift_results_with_summary = drift_analyzer.add_summary_column(
            metrics_to_consider=drift_metrics_to_consider
        )
        semaphore_kpis = drift_analyzer.calculate_semaphore_kpis('PSI')
        semaphore_config_drift = drift_analyzer.get_drift_semaphore_config()
        variable_comparison = drift_analyzer.variable_comparison if hasattr(drift_analyzer, 'variable_comparison') else None

        significant_drift = (drift_results['KS p-value'] < 0.05).sum()
        avg_psi = drift_results['PSI'].mean()
        max_psi = drift_results['PSI'].max()

        logger.info(f"  Drift significativo: {significant_drift} vars | PSI avg: {avg_psi:.4f} | PSI max: {max_psi:.4f}")

        if semaphore_kpis:
            logger.info(f"  🟢 Verde: {semaphore_kpis.get('verde', 0)} | 🟡 Amarillo: {semaphore_kpis.get('amarillo', 0)} | 🔴 Rojo: {semaphore_kpis.get('rojo', 0)}")
        else:
            logger.warning("No se obtuvieron resultados de drift")

except Exception as e:
    logger.log_exception(operation='drift_analysis', exception=e, should_raise=True)

In [0]:
# =============================================================================
# GUARDAR METADATOS DE DRIFT
# =============================================================================
try:
    logger.info("GUARDANDO METADATOS DE DRIFT")
    save_start = time.time()

    metadata_dir = os.path.join(TEMP_REPORTS_OUTPUT_PATH, "metadata")
    os.makedirs(metadata_dir, exist_ok=True)

    serializer = MetadataSerializer(metadata_dir)

    drift_metadata_path = None
    if drift_results_with_summary is not None and not drift_results_with_summary.empty:
        drift_metadata_path = serializer.serialize_drift_metrics(
            drift_results=drift_results_with_summary,
            variable_comparison=variable_comparison,
            semaphore_kpis=semaphore_kpis,
            semaphore_config=semaphore_config_drift,
            filename="drift_metrics.json"
        )
        logger.info(f" ✓ Drift metrics: {drift_metadata_path}")

    drift_bundle = {
        "metadata": {
            "mes_vta": str(MES_VTA),
            "fecha_generacion": datetime.now().isoformat(),
            "environment": ENV,
            "table_name": OUTPUT_TABLE,
            "total_records_actual": actual_count,
            "total_records_historico": historico_count,
            "meses_historicos": len(meses_historicos),
            "meses_historicos_list": [str(m) for m in meses_historicos],
            "variables_count": len(variables_existentes),
        },
        "drift_summary": {
            "significant_drift_vars": int(significant_drift),
            "avg_psi": float(avg_psi) if avg_psi else 0,
            "max_psi": float(max_psi) if max_psi else 0,
            "semaphore_kpis": semaphore_kpis if semaphore_kpis else {},
        }
    }

    drift_bundle_path = os.path.join(metadata_dir, "drift_metadata.json")
    with open(drift_bundle_path, 'w', encoding='utf-8') as f:
        json.dump(drift_bundle, f, indent=2, ensure_ascii=False, default=str)
    logger.info(f" ✓ Drift bundle: {drift_bundle_path}")

    save_duration = time.time() - save_start

except Exception as e:
    logger.log_exception(operation='save_drift_metadata', exception=e, should_raise=True)

In [0]:
# =============================================================================
# LIBERAR CACHE + RESUMEN FINAL
# =============================================================================
try:
    df_actual.unpersist()
    df_historico.unpersist()
except:
    pass

total_duration = time.time() - start_time

logger.info("=" * 60)
logger.info("ENGINE DRIFT COMPLETADO")
logger.info("=" * 60)
logger.info(f"Duratión total: {total_duration:.2f}s")
logger.info(f"  Carga: {load_duration:.1f}s")
logger.info(f"  Drift: {drift_duration:.1f}s")
logger.info(f"  Metadatos: {save_duration:.1f}s")
logger.info(f"Variables: {len(variables_existentes)}")

logger.log_stage_end('engine_drift', duration=total_duration,
                    records_processed=actual_count,
                    variables_analyzed=len(variables_existentes))